# CU-BEMS Intelligence Suite — Question 1 (Operational Baseload)
## Notebook 1C: Data Cleaning & Signal Rectification (`qn1c_data_cleaning.ipynb`)

---

### Core Cleaning Philosophy: Physics & Metrology Integrity
1. **Zero-Power Preservation (Crucial)**:
   * A reading of `0.000 kW` indicates a luminaire or equipment circuit is fully de-energized.
   * We strictly preserve `0.000 kW` and never treat zeroes as missing data.
2. **Strictly No Artificial Backfilling or Forward-Filling**:
   * Imputing missing electrical power measurements fabricates artificial kilowatt-hours.
   * Unrecorded minutes are retained as `NaN` and safely excluded via `skipna=True` during energy integration, ensuring all financial savings are conservative, lower-bound estimates.
3. **Non-Negative Active Power Clipping**:
   * Passive circuits occasionally register minute negative jitter (`-0.003 kW`) due to reactive sensor noise. We clip lower bounds to `0.0 kW`.
4. **Timestamp Parsing & Deduplication**:
   * Duplicate polling intervals are removed, and timestamps are structured as datetime indices.


In [1]:
# ==============================================================================
# 1. ENVIRONMENT & DATA DIRECTORY INITIALIZATION
# ==============================================================================
import os
import time
import pandas as pd
import numpy as np

notebook_dir = os.getcwd()
data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "data"))
print(f"[OK] Cleaning engine ready.\n  Data Cache: {data_dir}")

[OK] Cleaning engine ready.
  Data Cache: /Users/jiaxin/desk/IS4001-G1T4/data


We load each floor's raw electrical telemetry, deduplicate identical intervals, clip negative reactive noise, and preserve genuine measurement gaps without artificial data fabrication.

In [2]:
# ==============================================================================
# 2. EXECUTE SIGNAL RECTIFICATION ACROSS ALL 7 FLOORS
# ==============================================================================
clean_floor_data = {}
cleaning_metrics = []

t_clean_start = time.time()

for floor_num in range(1, 8):
    floor_key = f"Floor{floor_num}"
    raw_file = os.path.join(data_dir, f"raw_{floor_key}.csv")
    
    if not os.path.exists(raw_file):
        print(f"[ERROR] Raw file not found: {raw_file}. Please run qn1b_raw_ingestion.ipynb first.")
        continue
        
    df = pd.read_csv(raw_file, low_memory=False)
    raw_count = len(df)
    
    # 1. Datetime Index Parsing & Null Date Dropping
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.dropna(subset=['Date'])
    df = df.set_index('Date').sort_index()
    
    # 2. Polling Deduplication
    dup_count = df.index.duplicated(keep='first').sum()
    df = df[~df.index.duplicated(keep='first')]
    
    # Identify electrical circuits
    elec_cols = [c for c in df.columns if ('_Light(kW)' in c) or ('_Plug(kW)' in c)]
    total_elec_cells = len(df) * len(elec_cols)
    nan_before = df[elec_cols].isna().sum().sum()
    elec_nan_pct_before = (nan_before / total_elec_cells) * 100 if total_elec_cells > 0 else 0
    
    # 3. Non-Negative Active Power Clipping
    df[elec_cols] = df[elec_cols].clip(lower=0.0)
    
    # 4. Natural Metrology Preservation (No Artificial Forward/Backfilling)
    # Missing minutes remain NaN so we never fabricate energy consumption
    nan_after = df[elec_cols].isna().sum().sum()
    elec_nan_pct_after = (nan_after / total_elec_cells) * 100 if total_elec_cells > 0 else 0
    
    # Retain strictly electrical circuits (Lights and Plugs only, dropping any non-electrical columns)
    df = df[elec_cols]
    clean_floor_data[floor_key] = df
    
    cleaning_metrics.append({
        'Floor': floor_key,
        'Raw_Records': raw_count,
        'Duplicates_Removed': dup_count,
        'Clean_Timestamps': len(df),
        'Elec_Circuits': len(elec_cols),
        'Elec_NaN_Before_%': round(elec_nan_pct_before, 2),
        'Elec_NaN_After_%': round(elec_nan_pct_after, 2),
        'Completeness_%': round(100.0 - elec_nan_pct_after, 2)
    })

audit_df = pd.DataFrame(cleaning_metrics)
print(f"[SUCCESS] Cleaned all 7 floors in {time.time() - t_clean_start:.2f} seconds.")
display(audit_df)


[SUCCESS] Cleaned all 7 floors in 4.37 seconds.


,Floor,Raw_Records,Duplicates_Removed,Clean_Timestamps,Elec_Circuits,Elec_NaN_Before_%,Elec_NaN_After_%,Completeness_%
0,Floor1,790560,0,790560,7,0.73,0.73,99.27
1,Floor2,790560,0,790560,8,0.17,0.17,99.83
2,Floor3,790560,0,790560,10,7.70,7.70,92.30
3,Floor4,790560,0,790560,10,6.23,6.23,93.77
4,Floor5,790560,0,790560,10,1.82,1.82,98.18
5,Floor6,689129,0,689128,10,0.21,0.21,99.79
6,Floor7,790560,0,790560,10,0.49,0.49,99.51


## 3. Physical Sanity & Zero-Preservation Verification
We confirm that de-energized circuits (`0.0 kW`) have been preserved as true zero-watt values.

In [3]:
# ==============================================================================
# 3. ZERO-PRESERVATION CHECK (FLOOR 1 SAMPLE)
# ==============================================================================
f1 = clean_floor_data['Floor1']
f1_lights = [c for c in f1.columns if '_Light(kW)' in c]

print("--- ZERO-POWER READING AUDIT: FLOOR 1 LIGHTING CIRCUITS ---")
for c in f1_lights:
    zero_count = (f1[c] == 0.0).sum()
    near_zero = (f1[c] < 0.02).sum()
    print(f"{c:22s} | Exact 0.0 kW: {zero_count:,} ({zero_count/len(f1)*100:.1f}%) | <0.02 kW: {near_zero:,} ({near_zero/len(f1)*100:.1f}%)")

print("\n[VERIFIED] True physical zero-load readings are preserved intact for accurate baseload modeling.")

--- ZERO-POWER READING AUDIT: FLOOR 1 LIGHTING CIRCUITS ---
z1_Light(kW)           | Exact 0.0 kW: 97,759 (12.4%) | <0.02 kW: 283,924 (35.9%)
z2_Light(kW)           | Exact 0.0 kW: 10 (0.0%) | <0.02 kW: 26 (0.0%)
z3_Light(kW)           | Exact 0.0 kW: 6 (0.0%) | <0.02 kW: 28 (0.0%)
z4_Light(kW)           | Exact 0.0 kW: 165 (0.0%) | <0.02 kW: 1,591 (0.2%)

[VERIFIED] True physical zero-load readings are preserved intact for accurate baseload modeling.


## 4. Persisting Clean Telemetry Artifacts
We save each cleaned floor series into `../data/clean_Floor{1-7}.csv`.

In [4]:
# ==============================================================================
# 4. SAVE CLEANED TELEMETRY
# ==============================================================================
for floor_key, df in clean_floor_data.items():
    clean_path = os.path.join(data_dir, f"clean_{floor_key}.csv")
    df.to_csv(clean_path)
    print(f"[SAVED] {floor_key} -> {clean_path} ({len(df):,} records)")

print("\n[SUMMARY] Data cleaning complete. Proceed to 'qn1d_schedule_merge.ipynb'.")

[SAVED] Floor1 -> /Users/jiaxin/desk/IS4001-G1T4/data/clean_Floor1.csv (790,560 records)


[SAVED] Floor2 -> /Users/jiaxin/desk/IS4001-G1T4/data/clean_Floor2.csv (790,560 records)


[SAVED] Floor3 -> /Users/jiaxin/desk/IS4001-G1T4/data/clean_Floor3.csv (790,560 records)


[SAVED] Floor4 -> /Users/jiaxin/desk/IS4001-G1T4/data/clean_Floor4.csv (790,560 records)


[SAVED] Floor5 -> /Users/jiaxin/desk/IS4001-G1T4/data/clean_Floor5.csv (790,560 records)


[SAVED] Floor6 -> /Users/jiaxin/desk/IS4001-G1T4/data/clean_Floor6.csv (689,128 records)


[SAVED] Floor7 -> /Users/jiaxin/desk/IS4001-G1T4/data/clean_Floor7.csv (790,560 records)

[SUMMARY] Data cleaning complete. Proceed to 'qn1d_schedule_merge.ipynb'.
